# الفصل الخامس عشر: Type Hints — نحو برمجة أكثر وضوحًا ودقة (باستخدام mypy)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AhmedAlmaghz/Python-Book/blob/main/dist/notebooks/chapter-15.ipynb)

> **الوقت المتوقع:** ~18 دقيقة قراءة · **18** قسمًا · **50** مثالًا برمجيًا.

**كيف تستخدم هذا الدفتر؟**

1. شغّل الخلايا بالترتيب من الأعلى إلى الأسفل (`Shift` + `Enter`).
2. خلايا الشيفرة قابلة للتعديل: غيّر القيم وأعد التشغيل لتجرّب.
3. خلايا 🧾 معروضة للقراءة فقط (أوامر طرفية، أو إدخال يدوي، أو مقتطف ناقص)، وخلايا ⚠️ تُظهر خطأً مقصودًا يُلتقط تلقائيًا فلا يتوقف «تشغيل الكل».

> **Kaggle:** نزّل الملف من المستودع ثم `File ▸ Import Notebook`، أو استورده مباشرة من GitHub بعد ربط حسابك.

<details>
<summary><b>محتويات الفصل</b></summary>

- **15.1** لماذا يستحق هذا الفصل اهتمامك الكامل؟
- **15.2** مراجعة سريعة: أين وصلنا حتى الآن؟
- **15.3** الأنواع المركّبة (Generic Types)
- **15.4** القيم الاختيارية: Optional و None
- **15.5** عندما يكون هناك أكثر من نوع ممكن: Union
- **15.6** النوع الهارب: Any
- **15.7** الدوال كقيم: Callable
- **15.8** القيم الثابتة المحدودة: Literal
- **15.9** القواميس المُهيكَلة: TypedDict
- **15.10** البرمجة العامة (Generics) باستخدام TypeVar
- **15.11** البروتوكولات: Protocol (Duck Typing المُهيكَل)
- **15.12** أداة mypy: التحقق الساكن من الأنواع
- **15.13** أفضل الممارسات العامة
- **15.14** مشروع تطبيقي: نظام إدارة مهام مُنمَّط بالكامل
- **15.15** تمارين
- **15.16** اختبر فهمك (Quiz)
- **15.17** ملخص الفصل (Cheat Sheet)
- **15.18** ماذا بعد؟

</details>

In [ ]:
#@title ⚙️ تهيئة البيئة  { display-mode: "form" }
import sys, os, random, pathlib

print("Python:", sys.version.split()[0], "|", sys.platform)
assert sys.version_info >= (3, 10), "هذا الكتاب يفترض Python 3.10 أو أحدث"

SEED = 42
random.seed(SEED)

# تثبيت الحزم الناقصة فقط (سريع إن كانت مثبّتة أصلًا)
import importlib.util as _iu
_need = [m for m in ["mypy"] if _iu.find_spec(m) is None]
if _need:
    %pip install -q mypy

> ℹ️ هذا الفصل يستخدم `mypy` للفحص الساكن. أول خلية تثبّته إن لم يكن موجودًا.

## 15.1 لماذا يستحق هذا الفصل اهتمامك الكامل؟

تخيّل نفسك ضمن فريق مكوّن من خمسة مبرمجين يعملون على مشروع ضخم. يكتب أحد زملائك دالة باسم `calculate_discount`، وتحتاج إلى استخدامها في كودك. تفتح تعريف الدالة فتجد:

In [ ]:
def calculate_discount(price, percentage, customer):
    ...

تتوالى الأسئلة في ذهنك فورًا:

- هل `price` عدد صحيح (`int`) أم عشري (`float`)؟
- هل `percentage` قيمة بين 0 و1 (مثل 0.2) أم بين 0 و100 (مثل 20)؟
- ما طبيعة `customer`؟ نص يمثّل اسمًا؟ كائن من فئة `Customer`؟ أم قاموس يحمل بيانات العميل؟
- وماذا تُرجع الدالة في النهاية؟ رقمًا؟ نصًا يمثّل الفاتورة؟

في لغة ديناميكية التنميط مثل Python، لا تجد إجابة قاطعة إلا بقراءة جسم الدالة كاملًا، أو بتجربتها وانتظار الخطأ. هذا يُهدر وقتك الثمين، ويفتح الباب أمام أخطاء قد لا تظهر إلا في بيئة الإنتاج (Production)، بعد أن يعتمد عليها آلاف المستخدمين.

والآن، تخيّل أن الدالة كُتبت على هذا النحو:

```python
def calculate_discount(price: float, percentage: float, customer: Customer) -> float:
    ...
```

🧾 **للقراءة فقط** — مثال مقتطف يعتمد على `Customer` من مثال سابق.

بمجرد النظر إلى التوقيع (Signature)، اتضح لك كل شيء: الأسعار أرقام عشرية، النسبة رقم عشري، العميل كائن من فئة `Customer`، والناتج رقم عشري. لا حاجة لقراءة جسم الدالة، ولا مجال للتخمين.

هذا هو جوهر **Type Hints** (تلميحات الأنواع): توثيق حيّ ودقيق لكودك، يفهمه الإنسان والآلة معًا.

منذ الفصل الثالث، ونحن نكتب عبارات من هذا القبيل في كل مثال تقريبًا:

In [ ]:
age: int = 25
name: str = "محمد"

def greet(name: str) -> str:
    return f"مرحبًا، {name}!"

كنتَ تستخدمها دون الخوض في تفاصيلها. حان الوقت الآن لفهمها بعمق حقيقي: ما طبيعتها فعليًا؟ كيف تُستخدم مع هياكل البيانات المعقّدة؟ وكيف تُسخّر أداة باسم **mypy** للتحقق الآلي من كودك، واكتشاف الأخطاء قبل تشغيل البرنامج أصلًا؟

> 💡 **نصيحة الخبير:** لا تعتبر Type Hints «رفاهية للمحترفين فقط». شركات عملاقة مثل Google وDropbox وInstagram تعتمد عليها بكثافة في أكواد Python الإنتاجية، لأنها تقلّص الأخطاء بشكل ملموس، وتُمكّن أدوات مثل VS Code من اقتراح الكود الصحيح تلقائيًا (Autocomplete أدق بكثير).

## 15.2 مراجعة سريعة: أين وصلنا حتى الآن؟

منذ الفصل الثالث، تعاملنا مع صيغة بسيطة ومباشرة:

In [ ]:
# متغيرات
age: int = 25
price: float = 19.99
name: str = "سارة"
is_active: bool = True

# دوال
def add(a: int, b: int) -> int:
    return a + b

def greet(name: str) -> None:
    print(f"مرحبًا، {name}")

**القواعد الأساسية التي نعرفها:**

| العنصر | الصيغة | مثال |
|---|---|---|
| متغير | `name: Type = value` | `age: int = 25` |
| معامل دالة | `param: Type` | `def f(x: int):` |
| قيمة الإرجاع | `-> Type` | `def f() -> str:` |
| دالة لا تُرجع شيئًا | `-> None` | `def f() -> None:` |

لكن حان وقت السؤال الجوهري الذي ربما يراودك منذ البداية:

### هل تتحقق Python فعليًا من هذه الأنواع؟

الجواب القاطع: **لا، إطلاقًا**. جرّب هذا الكود:

In [ ]:
def add(a: int, b: int) -> int:
    return a + b

result = add("مرحبًا", "!")  # لا يوجد أي خطأ من Python!
print(result)  # "مرحبًا!"

شغّل هذا الكود، وستجده يعمل دون أدنى مشكلة! فسّرت Python العملية `+` على أنها دمج نصوص (Concatenation)، لأن `a` و`b` نصوص فعليًا وقت التشغيل، وتجاهلت التلميحة `int` تمامًا.

> ⚠️ **تحذير مهم — سوء فهم شائع:** Type Hints في Python مجرد تلميحات (Hints)، وليست آلية فرض (Enforcement). يتجاهلها المُفسِّر (Interpreter) تمامًا وقت التشغيل (باستثناء حالات نادرة متقدمة). وظيفتها الحقيقية تنحصر في:
> - **القارئ البشري** (أنت، زملاؤك، أنت نفسك بعد ستة أشهر).
> - **الأدوات الخارجية**، مثل المحرر (VS Code/Pylance) أو أداة تحقق ساكنة مثل mypy.
>
> هذا يختلف جذريًا عن لغات مثل Java أو C# أو TypeScript، حيث يرفض المترجم (Compiler) تشغيل الكود إن كانت الأنواع غير متطابقة.

### 🔄 مقارنة: Python مقابل لغات ذات تنميط ساكن حقيقي

| الخاصية | Python (مع Type Hints) | Java / C# / TypeScript |
|---|---|---|
| من يتحقق من الأنواع؟ | أدوات خارجية اختيارية (mypy) | المترجم نفسه، إلزاميًا |
| هل يعمل الكود رغم خطأ في النوع؟ | نعم (وقد ينهار لاحقًا) | لا، يرفض المترجم البناء أصلًا |
| هل Type Hints إلزامية؟ | لا، اختيارية 100% | نعم (أو شبه إلزامية) |
| الاسم التقني | Gradual Typing (تنميط تدريجي) | Static Typing (تنميط ساكن) |

يقودنا هذا إلى سؤال محوري: كيف نستفيد إذًا من الفائدة الحقيقية لـ Type Hints، ما دامت Python تتجاهلها؟

الجواب: نستعين بأداة اسمها **mypy** تقرأ كودك قبل التشغيل وتتحقق من تطابق الأنواع، ثم تُبلغك بأي أخطاء عبر رسائل نصية واضحة. سنتناولها بالتفصيل بعد أن نُتقن أولًا كتابة الأنواع نفسها.

## 15.3 الأنواع المركّبة (Generic Types)

تعاملنا حتى الآن مع أنواع بسيطة: `int`, `float`, `str`, `bool`, `None`. لكن ماذا عن القوائم والقواميس؟

### القوائم: `list[T]`

In [ ]:
# قائمة أعداد صحيحة
numbers: list[int] = [1, 2, 3, 4, 5]

# قائمة نصوص
names: list[str] = ["أحمد", "سارة", "خالد"]

# قائمة متداخلة (قائمة من قوائم أعداد)
matrix: list[list[int]] = [[1, 2], [3, 4], [5, 6]]

`T` هنا رمز اصطلاحي يعني «أي نوع»، لكن في الاستخدام الفعلي نكتب النوع الحقيقي مثل `int` أو `str`.

> 💡 **نصيحة تاريخية:** في إصدارات Python القديمة (قبل 3.9)، كان لا بدّ من استيراد `List` من وحدة `typing`:
> ```python
> from typing import List
> numbers: List[int] = [1, 2, 3]
> ```
> منذ Python 3.9، أصبح بالإمكان استخدام `list` مباشرة بحرف صغير دون استيراد أي شيء. وبما أننا نستهدف Python 3.14 في هذا الكتاب، سنعتمد دائمًا الصيغة الحديثة (`list[int]` وليس `List[int]`).

### الصناديق (Tuples): `tuple[T1, T2, ...]`

تتميّز الصناديق (Tuples) بأن كل عنصر فيها قد يكون من نوع مختلف، وعددها غالبًا ثابت:

In [ ]:
# صندوق يحتوي اسمًا (نص) وعمرًا (عدد صحيح)
person: tuple[str, int] = ("ليلى", 28)

# صندوق إحداثيات (x, y, z) - ثلاثة أعداد عشرية
point: tuple[float, float, float] = (1.5, 2.0, 3.5)

وإذا أردت صندوقًا بعدد متغيّر من العناصر، كلها من النوع نفسه، استخدم `...`:

In [ ]:
# صندوق بعدد غير محدد من الأعداد الصحيحة
scores: tuple[int, ...] = (90, 85, 100, 70, 60)

### القواميس: `dict[K, V]`

In [ ]:
# قاموس: المفتاح نص، القيمة عدد صحيح
ages: dict[str, int] = {"أحمد": 30, "سارة": 25}

# قاموس أكثر تعقيدًا: المفتاح نص، القيمة قائمة نصوص
courses_by_student: dict[str, list[str]] = {
    "أحمد": ["رياضيات", "فيزياء"],
    "سارة": ["كيمياء"],
}

### المجموعات: `set[T]`

In [ ]:
unique_ids: set[int] = {101, 102, 103}
tags: set[str] = {"python", "برمجة", "تعليم"}

### مثال شامل يجمع كل العناصر السابقة

In [ ]:
def summarize_students(
    grades: dict[str, list[int]]
) -> dict[str, float]:
    """
    يستقبل قاموسًا بأسماء الطلاب ودرجاتهم،
    ويُرجع قاموسًا بأسماء الطلاب ومتوسط درجاتهم.
    """
    averages: dict[str, float] = {}
    for student, scores in grades.items():
        averages[student] = sum(scores) / len(scores)
    return averages

student_grades: dict[str, list[int]] = {
    "محمد": [85, 90, 78],
    "فاطمة": [92, 88, 95],
}

result: dict[str, float] = summarize_students(student_grades)
print(result)
# {'محمد': 84.33333333333333, 'فاطمة': 91.66666666666667}

### 🔄 مقارنة داخل Python: هل أكتب Type Hints لكل متغير؟

لاحظ أننا لم نكتب `student: str` و`scores: list[int]` داخل الحلقة `for student, scores in grades.items():`، وهذا مقصود وصحيح تمامًا. فعندما يكون نوع المتغير واضحًا من السياق (هنا، واضح من نوع `grades` نفسه)، فإن أدوات مثل mypy وPylance تستنتج (Infer) النوع تلقائيًا، ولا حاجة لتكرار كتابته. يُسمى هذا **Type Inference**. القاعدة الذهبية: اكتب Type Hints عند توقيعات الدوال والمتغيرات المعقّدة، ودع الأداة تستنتج الباقي بنفسها.

## 15.4 القيم الاختيارية: Optional و None

من أشهر الأخطاء في أي لغة برمجة محاولة التعامل مع قيمة `None` (أو `null` في لغات أخرى) وكأنها قيمة حقيقية، فينهار البرنامج بخطأ شهير اسمه في Python:

```
AttributeError: 'NoneType' object has no attribute '...'
```

### المشكلة بدون Type Hints

In [ ]:
def find_user(user_id: int, users: dict[int, str]) -> str:
    return users.get(user_id)  # قد تُرجع None إن لم يوجد المستخدم!

هنا قد تُرجع `users.get(user_id)` قيمة `None` إن لم يكن `user_id` موجودًا، بينما يقول التوقيع `-> str`! هذا كذب في التوثيق قد يُضلل كل من يستخدم الدالة لاحقًا.

### الحل: `Optional[T]` أو الصيغة الحديثة `T | None`

In [ ]:
from typing import Optional

def find_user_old_style(user_id: int, users: dict[int, str]) -> Optional[str]:
    return users.get(user_id)

لكن منذ Python 3.10، أصبح بإمكاننا استخدام الرمز `|` (Pipe) مباشرة، وهو الأسلوب الموصى به حاليًا لأنه أقصر وأوضح ولا يتطلب استيرادًا:

In [ ]:
def find_user(user_id: int, users: dict[int, str]) -> str | None:
    return users.get(user_id)

result: str | None = find_user(5, {1: "أحمد", 2: "سارة"})

if result is not None:
    print(result.upper())  # آمن تمامًا، لأننا تحققنا أولًا
else:
    print("المستخدم غير موجود")

تعني `str | None` حرفيًا: "إمّا نص، أو `None`". وهذا بالضبط ما يعنيه `Optional[str]` — فـ `Optional[T]` ليست سوى اختصار تاريخي لـ `T | None`.

> ⚠️ **خطأ شائع جدًا:** لا تخلط بين `Optional[str]` ومعامل اختياري له قيمة افتراضية. هذان مفهومان مختلفان تمامًا!

In [ ]:
# معامل اختياري (له قيمة افتراضية) - لا علاقة له بـ Optional type
def greet(name: str = "ضيف") -> str:
    return f"مرحبًا، {name}"

# Optional type - القيمة قد تكون None
def find(key: str) -> str | None:
    ...

ويمكن أن يجتمع المفهومان معًا:

In [ ]:
def greet(name: str | None = None) -> str:
    if name is None:
        name = "ضيف"
    return f"مرحبًا، {name}"

## 15.5 عندما يكون هناك أكثر من نوع ممكن: Union

ماذا لو كانت دالة تقبل إما عددًا صحيحًا أو نصًا؟ هذا شائع جدًا، كما في حالة قراءة بيانات من مصدر خارجي (JSON، مدخلات المستخدم).

### الصيغة الحديثة (موصى بها من 3.10+): `|`

In [ ]:
def format_id(value: int | str) -> str:
    """يحوّل معرّفًا (رقمًا أو نصًا) إلى نص منسّق."""
    return f"ID-{value}"

print(format_id(42))      # "ID-42"
print(format_id("A001"))  # "ID-A001"

### الصيغة التقليدية (لا تزال شائعة في أكواد قديمة): `Union`

In [ ]:
from typing import Union

def format_id(value: Union[int, str]) -> str:
    return f"ID-{value}"

> 💡 **نصيحة الخبير:** بما أننا اعتمدنا Python 3.14 في هذا الكتاب، استخدم دائمًا `|` بدلًا من `Union`، فهي أقصر، ومدمجة في اللغة، ولا تحتاج استيرادًا. ستجد `Union` منتشرة في أكواد ومكتبات أقدم، لذا من المهم أن تتعرّف عليها، لكن تجنّب استخدامها في كودك الجديد.

### التعامل الآمن مع Union باستخدام isinstance

عندما يكون لديك متغير بنوع مُركّب، غالبًا ما تحتاج للتحقق من نوعه الفعلي قبل التعامل معه:

In [ ]:
def process(value: int | str) -> None:
    if isinstance(value, int):
        print(f"رقم مضاعف: {value * 2}")
    else:
        print(f"نص بأحرف كبيرة: {value.upper()}")

تُسمّى هذه التقنية **Type Narrowing** (تضييق النوع): تبدأ بنوع عام (`int | str`)، وبعد التحقق بـ `isinstance`، تتعرّف الأداة (mypy/Pylance) داخل كل فرع من `if` على النوع الدقيق، وتمنحك اقتراحات Autocomplete صحيحة.

## 15.6 النوع الهارب: Any

أحيانًا، لا تستطيع (أو لا ترغب) في تحديد نوع دقيق. هنا يأتي دور `Any`:

In [ ]:
from typing import Any

def process_anything(data: Any) -> Any:
    return data

تعني `Any` حرفيًا: «عطّل التحقق من النوع هنا تمامًا». أي قيمة مقبولة، وأي عملية عليها مقبولة، دون أي تحذير من mypy.

> ⚠️ **تحذير خطير:** تُعدّ `Any` بمثابة «ثقب أسود» في نظام الأنواع. كل ما تلمسه `Any` يفقد الحماية. لا تستخدمها إلا في الحالات التالية:
> - عند التعامل مع بيانات حقيقية غير معروفة الشكل (مثل نتيجة `json.loads()` من مصدر خارجي).
> - أثناء ترحيل كود قديم تدريجيًا إلى Type Hints (كحل مؤقت).
>
> لا تستخدمها أبدًا كحل كسول لتفادي التفكير في النوع الصحيح. فكل استخدام لـ `Any` فرصة ضائعة للاستفادة الحقيقية من Type Hints.

### 🔄 مقارنة: Any مقابل object

كلاهما يقبل أي قيمة، لكن الفرق بينهما جوهري:

In [ ]:
def func_any(x: Any) -> None:
    x.anything_at_all()  # لا تحذير من mypy حتى لو الدالة غير موجودة!

def func_object(x: object) -> None:
    x.anything_at_all()  # mypy يُصدر خطأً! object لا يملك هذه الدالة

`object` هو النوع الأب لكل الأنواع في Python (صحيح ونوع آمن)، لكنه لا يسمح باستدعاء دوال خاصة دون التحقق أولًا (`isinstance`)، بينما `Any` يُلغي أي تحقق كليًا. القاعدة: فضّل `object` على `Any` كلما أمكن ذلك.

## 15.7 الدوال كقيم: Callable

تذكر من الفصلين العاشر والثاني عشر أن الدوال في Python كائنات من الدرجة الأولى (First-Class Citizens) — يمكن تمريرها كمعاملات. فكيف نكتب Type Hint لمعامل يكون دالة؟

In [ ]:
from typing import Callable

def apply_operation(
    numbers: list[int],
    operation: Callable[[int], int]
) -> list[int]:
    """يطبّق 'operation' على كل عنصر في القائمة."""
    return [operation(n) for n in numbers]

def double(x: int) -> int:
    return x * 2

def square(x: int) -> int:
    return x * x

print(apply_operation([1, 2, 3], double))  # [2, 4, 6]
print(apply_operation([1, 2, 3], square))  # [1, 4, 9]

# يعمل أيضًا مع lambda
print(apply_operation([1, 2, 3], lambda x: x + 10))  # [11, 12, 13]

صيغة `Callable[[Arg1Type, Arg2Type, ...], ReturnType]`:

- داخل القوسين الأولين `[...]`: قائمة بأنواع المعاملات.
- بعد الفاصلة: نوع القيمة المُرجعة.

In [ ]:
# دالة بدون معاملات تُرجع نصًا
greeting_provider: Callable[[], str]

# دالة تستقبل int و str وتُرجع bool
validator: Callable[[int, str], bool]

# دالة بأي عدد ونوع من المعاملات (تجنّب هذا إن أمكن، لكنه موجود)
from typing import Callable
any_callable: Callable[..., int]  # ... تعني "أي معاملات"

### مثال عملي: Decorator موصوف بالكامل بـ Type Hints

In [ ]:
from typing import Callable
import time
import functools

def measure_time(func: Callable[..., float]) -> Callable[..., float]:
    @functools.wraps(func)
    def wrapper(*args: int, **kwargs: int) -> float:
        start = time.perf_counter()
        result = func(*args, **kwargs)
        elapsed = time.perf_counter() - start
        print(f"استغرقت {func.__name__}: {elapsed:.4f} ثانية")
        return result
    return wrapper

@measure_time
def slow_calculation(n: int) -> float:
    return sum(i ** 2 for i in range(n)) / n

slow_calculation(1_000_000)

> 💡 **نصيحة الخبير:** كتابة Type Hints دقيقة للـ Decorators (خصوصًا مع `*args` و`**kwargs`) من أصعب الجوانب في نظام الأنواع. في المشاريع الكبيرة، تُستخدم تقنيات متقدمة مثل `ParamSpec` و`TypeVar` معًا لحفظ توقيع الدالة الأصلية بدقة. لا داعي للقلق بشأن هذا التعقيد الآن — في أغلب الأحيان، التوقيع البسيط كافٍ تمامًا لمشاريعك.

## 15.8 القيم الثابتة المحدودة: Literal

أحيانًا، لا يكفي القول «هذا نص» — بل تريد التصريح بأن هذا النص يجب أن يكون بالضبط واحدًا من قيم محددة سلفًا. هنا يأتي دور `Literal`:

```python
from typing import Literal

def set_status(status: Literal["active", "inactive", "pending"]) -> None:
    print(f"الحالة الجديدة: {status}")

set_status("active")    # صحيح
set_status("pending")   # صحيح
set_status("deleted")   # ❌ mypy يرفض هذا! "deleted" ليست من القيم المسموحة
```

⚠️ **للقراءة فقط** — خطأ مقصود للتوضيح، لذا لا يُنفَّذ.

هذا أقوى بكثير من استخدام `str` العادي، لأن `str` تقبل أي نص على الإطلاق، بينما `Literal` تحصر الخيارات بدقة. يُستخدم غالبًا مع الثوابت المعروفة مسبقًا (حالات، أوضاع تشغيل، اتجاهات...).

```python
Direction = Literal["up", "down", "left", "right"]

def move(direction: Direction, steps: int = 1) -> tuple[int, int]:
    deltas: dict[Direction, tuple[int, int]] = {
        "up": (0, 1),
        "down": (0, -1),
        "left": (-1, 0),
        "right": (1, 0),
    }
    dx, dy = deltas[direction]
    return (dx * steps, dy * steps)

print(move("up", 3))  # (0, 3)
```

🧾 **للقراءة فقط** — مثال مقتطف يعتمد على `Literal` من مثال سابق.

لاحظ أننا عرّفنا `Direction` كاسم مستعار (Type Alias) للنوع المركّب، لتجنّب تكرار كتابة `Literal["up", "down", "left", "right"]` في كل مكان.

> 💡 **نصيحة الخبير:** يجعل تعريف أسماء مستعارة للأنواع المعقّدة (Type Aliases) كودك أنظف بكثير:
> ```python
> UserId = int
> UserName = str
> UserDatabase = dict[UserId, UserName]
>
> def get_user(db: UserDatabase, uid: UserId) -> UserName | None:
>     return db.get(uid)
> ```
> ومنذ Python 3.12، توجد صيغة مخصصة وأوضح لهذا الغرض باستخدام الكلمة المفتاحية `type`:
> ```python
> type UserDatabase = dict[int, str]
> ```

## 15.9 القواميس المُهيكَلة: TypedDict

تذكّر القواميس من الفصل التاسع: مرنة جدًا، لكن مرونتها قد تنطوي على خطر. فماذا لو أردت قاموسًا يمثّل «بطاقة موظف» بحقول محددة ومعروفة مسبقًا (`name`, `age`, `salary`)، دون اللجوء إلى فئة (Class) كاملة لهذا الغرض البسيط؟

```python
from typing import TypedDict

class Employee(TypedDict):
    name: str
    age: int
    salary: float

def print_employee(emp: Employee) -> None:
    print(f"{emp['name']} - {emp['age']} سنة - راتب {emp['salary']} دولار")

employee1: Employee = {
    "name": "ياسر",
    "age": 35,
    "salary": 5000.0,
}

print_employee(employee1)

# استخدام خاطئ يكتشفه mypy:
bad_employee: Employee = {
    "name": "سلمى",
    "age": "ثلاثون",  # ❌ خطأ! يجب أن يكون int وليس str
    "salary": 4500.0,
}
```

⚠️ **للقراءة فقط** — خطأ مقصود للتوضيح، لذا لا يُنفَّذ.

لاحظ أن `Employee` تبدو وكأنها فئة عادية تَرث من `TypedDict`، لكنها لا تُنشئ كائنات فعلية — عند التشغيل، `employee1` قاموس عادي تمامًا (`dict`). كل الفائدة هنا مخصصة فقط لأدوات التحقق الساكن مثل mypy.

### 🔄 مقارنة: متى أستخدم TypedDict ومتى أستخدم dataclass؟

| المعيار | TypedDict | @dataclass |
|---|---|---|
| النوع الفعلي وقت التشغيل | `dict` عادي | كائن (Object) حقيقي |
| الوصول للحقول | `emp["name"]` | `emp.name` |
| يحتاج JSON مباشرة (مثل API)؟ | ممتاز، لأنه `dict` بالأصل | يحتاج تحويلًا يدويًا |
| يحتاج دوالًا/methods إضافية؟ | لا يدعم ذلك | نعم، يدعم تمامًا |
| الأداء | أخف قليلًا | قريب جدًا |

القاعدة العملية: إذا كانت بياناتك ستُحوَّل مباشرة من/إلى JSON (كما سنرى كثيرًا في الفصل القادم ولاحقًا مع FastAPI)، فـ `TypedDict` مناسب جدًا. أما إذا كنت تبني نموذج بيانات (Domain Model) حقيقي بسلوك، فاستخدم `@dataclass` أو فئة كاملة.

## 15.10 البرمجة العامة (Generics) باستخدام TypeVar

هذا من أقوى مفاهيم Type Hints، ويُستخدم لكتابة دوال أو فئات تعمل مع أي نوع، مع الحفاظ على ترابط الأنواع بين المدخلات والمخرجات.

### المشكلة بدون Generics

In [ ]:
def get_first_element(items: list) -> Any:
    return items[0]

المشكلة: لا نعرف نوع العنصر الأول. فإن استدعيناها بقائمة أعداد صحيحة، يجب أن تعرف الأداة أن الناتج `int`، وإن استدعيناها بقائمة نصوص، يجب أن تعرف أن الناتج `str`. أما `Any` فتُفقدنا كل هذا الترابط.

### الحل: TypeVar

In [ ]:
from typing import TypeVar

T = TypeVar("T")

def get_first_element(items: list[T]) -> T:
    return items[0]

numbers: list[int] = [10, 20, 30]
first_number = get_first_element(numbers)  # mypy يستنتج: int

words: list[str] = ["كتاب", "قلم"]
first_word = get_first_element(words)  # mypy يستنتج: str

هنا `T` ليست نوعًا محددًا، بل متغير نوع (Type Variable) — «أيًا كان النوع الذي أعطيتني إياه، سأُرجع النوع نفسه». يُسمى هذا **Generic Programming** (البرمجة العامة).

### صيغة Python 3.12+ الحديثة (أبسط بكثير)

منذ Python 3.12، يمكن كتابة Generics مباشرة دون استيراد `TypeVar` صراحة، باستخدام صيغة جديدة أنظف:

In [ ]:
def get_first_element[T](items: list[T]) -> T:
    return items[0]

لاحظ الأقواس المربعة `[T]` مباشرة بعد اسم الدالة — هذا كل ما تحتاجه! هذه الصيغة أبسط وأقصر، وهي الموصى بها في Python 3.14.

### Generics مع الفئات (Classes)

يمكن أيضًا جعل فئة كاملة عامة (Generic Class). مثال: صندوق تخزين (Box) يحتفظ بعنصر واحد من أي نوع:

In [ ]:
class Box[T]:
    """صندوق يحتفظ بعنصر واحد من أي نوع."""

    def __init__(self, content: T) -> None:
        self.content = content

    def get(self) -> T:
        return self.content

    def __repr__(self) -> str:
        return f"Box({self.content!r})"

int_box = Box(42)           # mypy يستنتج: Box[int]
str_box = Box("مرحبًا")      # mypy يستنتج: Box[str]

print(int_box.get())  # 42
print(str_box.get())  # مرحبًا

value: int = int_box.get()  # صحيح، mypy يعرف أن الناتج int

### مثال عملي أكبر: مخزن بيانات عام (Generic Repository)

In [ ]:
class Repository[T]:
    """مخزن بسيط يدير مجموعة من العناصر من نوع T."""

    def __init__(self) -> None:
        self._items: list[T] = []

    def add(self, item: T) -> None:
        self._items.append(item)

    def get_all(self) -> list[T]:
        return self._items.copy()

    def find(self, predicate: Callable[[T], bool]) -> T | None:
        for item in self._items:
            if predicate(item):
                return item
        return None

from typing import Callable
from dataclasses import dataclass

@dataclass
class Product:
    name: str
    price: float

product_repo = Repository[Product]()
product_repo.add(Product("لابتوب", 1200.0))
product_repo.add(Product("هاتف", 800.0))

cheap_product = product_repo.find(lambda p: p.price < 1000)
print(cheap_product)  # Product(name='هاتف', price=800.0)

> 💡 **نصيحة الخبير:** قد تبدو Generics معقّدة في البداية، لكن استخدامها شائع جدًا في مكتبات حقيقية تستخدمها يوميًا: `list[int]` نفسها استخدام لـ Generic! فـ `list` فئة عامة تقبل أي نوع `T`. كلما فهمت Generics أكثر، فهمت لماذا تعمل هذه الأشياء بالطريقة التي تعمل بها.

## 15.11 البروتوكولات: Protocol (Duck Typing المُهيكَل)

تذكّر من الفصل الرابع عشر مفهوم Duck Typing: «إن كان يمشي كبطة ويصدر صوت بطة، فهو بطة» — أي أن Python لا تهتم بنوع الكائن الفعلي، بل بالقدرات (الدوال) التي يمتلكها.

لكن كيف نكتب Type Hint يُعبّر عن هذه الفكرة؟ أي: "أريد أي كائن يملك دالة `.draw()`"، بصرف النظر عن فئته الأصلية؟ هنا يأتي دور `Protocol`:

In [ ]:
from typing import Protocol

class Drawable(Protocol):
    def draw(self) -> str:
        ...

class Circle:
    def draw(self) -> str:
        return "رسم دائرة ⭕"

class Square:
    def draw(self) -> str:
        return "رسم مربع ⬜"

def render(shape: Drawable) -> None:
    print(shape.draw())

render(Circle())  # صحيح! Circle تملك draw()
render(Square())  # صحيح! Square تملك draw() أيضًا

لاحظ أن `Circle` و`Square` لا تَرثان من `Drawable` بأي شكل من الأشكال! يُسمى هذا **Structural Typing** (تنميط هيكلي): «إن كان الكائن يملك الدوال المطلوبة بنفس التوقيع، فهو يحقق البروتوكول تلقائيًا»، بخلاف الوراثة التقليدية (Nominal Typing) التي تتطلب إعلانًا صريحًا بالانتماء إلى فئة معينة.

### 🔄 مقارنة: Protocol مقابل ABC (Abstract Base Class)

| المعيار | Protocol | ABC (الفصل 14) |
|---|---|---|
| هل تحتاج وراثة صريحة؟ | لا | نعم، إلزامية |
| النمط | هيكلي (Structural) — «لو شكلك مطابق، تُقبل» | اسمي (Nominal) — «يجب أن تُعلن الانتماء» |
| الاستخدام الأنسب | عند التعامل مع فئات لا تتحكم بها (من مكتبات خارجية) | عند تصميم هرمية فئات خاصة بك من البداية |
| هل يُفرض فعليًا وقت التشغيل؟ | لا (إلا مع `@runtime_checkable`) | نعم، عبر `TypeError` عند `@abstractmethod` |

> 💡 **نصيحة الخبير:** استخدم `Protocol` عندما تكتب دالة تقبل «أي شيء يستطيع فعل كذا»، خصوصًا عند التعامل مع فئات من مكتبات لا تملك التحكم بها (فلا يمكنك جعلها تَرث من ABC خاصة بك). استخدم `ABC` عندما تُصمم هرمية فئات خاصة بك من الصفر وتريد فرض تنفيذ methods معينة.

## 15.12 أداة mypy: التحقق الساكن من الأنواع

وصلنا الآن إلى الجزء الأهم عمليًا: كيف نجعل كل هذه الـ Type Hints مفيدة فعليًا، لا مجرد توثيق جميل يُقرأ ويُتجاهل؟

### ما هي mypy؟

**mypy** أداة مستقلة (ليست جزءًا من Python نفسها) تقرأ ملفات كودك دون تشغيلها فعليًا، وتحلل كل Type Hints المكتوبة، وتبحث عن أي تضارب أو خلل منطقي في الأنواع، ثم تُصدر تقريرًا نصيًا بكل المشاكل التي رصدتها.

يُسمى هذا **Static Type Checking** (التحقق الساكن من الأنواع) — «ساكن» لأنه يحدث قبل تشغيل البرنامج، بخلاف الأخطاء التي لا تظهر إلا أثناء التشغيل (Runtime).

### تثبيت mypy

باستخدام `uv` (كما اعتدنا طوال الكتاب):

```bash
uv add --dev mypy
```

🧾 **للقراءة فقط** — أوامر `bash` تُنفَّذ من الطرفية، لا من الخلية.

نستخدم `--dev` لأن mypy أداة تطوير (Development Tool)، وليست جزءًا من المكتبات التي يحتاجها برنامجك فعليًا وقت التشغيل عند المستخدم النهائي.

### أول تجربة مع mypy

أنشئ ملفًا باسم `example.py`:

In [ ]:
def add(a: int, b: int) -> int:
    return a + b

result = add("مرحبًا", "!")
print(result)

ثم شغّل mypy على هذا الملف:

```bash
uv run mypy example.py
```

🧾 **للقراءة فقط** — أوامر `bash` تُنفَّذ من الطرفية، لا من الخلية.

**النتيجة:**

```
example.py:5: error: Argument 1 to "add" has incompatible type "str"; expected "int"  [arg-type]
example.py:5: error: Argument 2 to "add" has incompatible type "str"; expected "int"  [arg-type]
Found 2 errors in 1 file (checked 1 source file)
```

لاحظ شيئًا مذهلًا: لم نُشغّل البرنامج فعليًا! قرأ mypy الكود فقط، وأخبرنا بوجود خطأ قبل أي تشغيل. هذا الفرق جوهري — في مشروع كبير بآلاف الأسطر، يعني هذا اكتشاف أخطاء قد لا تظهر إلا بعد أشهر في الإنتاج، خلال ثوانٍ معدودة من كتابة الكود.

> 💡 **نصيحة الخبير:** إضافة Pylance في VS Code (التي ثبّتناها في الفصل الثاني) تُجري فعليًا تحليلًا مماثلًا لحظيًا أثناء الكتابة، وتُظهر الخط الأحمر المتعرج تحت الكود الخاطئ مباشرة. أما mypy، فيُستخدم عادة بالإضافة إلى ذلك كـ«بوابة فحص نهائية» ضمن CI/CD (عمليات تحقق آلية قبل دمج الكود)، لضمان عدم تجاهل أحد لتحذيرات المحرر.

### أمثلة أخطاء حقيقية تكتشفها mypy

**مثال 1: نسيان حالة None**

```python
def find_user(user_id: int, users: dict[int, str]) -> str | None:
    return users.get(user_id)

user = find_user(5, {1: "أحمد"})
print(user.upper())  # ❌ خطأ محتمل!
```

⚠️ **للقراءة فقط** — خطأ مقصود للتوضيح، لذا لا يُنفَّذ.

```bash
uv run mypy example.py
```

🧾 **للقراءة فقط** — أوامر `bash` تُنفَّذ من الطرفية، لا من الخلية.

```
error: Item "None" of "str | None" has no attribute "upper"  [union-attr]
```

يُخبرنا mypy: "`user` قد يكون `None`، و`None` لا يملك دالة `.upper()`". هذا بالضبط نوع الخطأ الذي كان سيظهر فقط عند استخدام `user_id` غير موجود، ربما بعد نشر التطبيق!

**التصحيح:**

In [ ]:
user = find_user(5, {1: "أحمد"})
if user is not None:
    print(user.upper())
else:
    print("المستخدم غير موجود")

**مثال 2: خطأ في عدد المعاملات**

```python
def greet(name: str, greeting: str = "مرحبًا") -> str:
    return f"{greeting}، {name}!"

print(greet())  # ❌ نسينا معاملًا إلزاميًا
```

⚠️ **للقراءة فقط** — خطأ مقصود للتوضيح، لذا لا يُنفَّذ.

```
error: Missing positional argument "name" in call to "greet"  [call-arg]
```

**مثال 3: استخدام خاطئ لنوع الإرجاع**

```python
def get_age() -> int:
    return "عشرون"  # ❌ نص بدل عدد صحيح
```

⚠️ **للقراءة فقط** — خطأ مقصود للتوضيح، لذا لا يُنفَّذ.

```
error: Incompatible return value type (got "str", expected "int")  [return-value]
```

🧾 **للقراءة فقط** — مقتطف غير مكتمل.

**مثال 4: خطأ في TypedDict**

```python
from typing import TypedDict

class Point(TypedDict):
    x: int
    y: int

p: Point = {"x": 1}  # ❌ ناقص حقل y
```

⚠️ **للقراءة فقط** — خطأ مقصود للتوضيح، لذا لا يُنفَّذ.

```
error: Missing key "y" for TypedDict "Point"  [typeddict-item]
```

🧾 **للقراءة فقط** — مقتطف غير مكتمل.

> ⚠️ **تحذير:** لا يستطيع mypy اكتشاف كل الأخطاء المنطقية، فهو يتحقق فقط من تطابق الأنواع وفق ما كتبته أنت من Type Hints. إن كتبت `Any` في كل مكان، فلن يجد mypy أي شيء يُبلّغ عنه! جودة فحص mypy تتناسب طرديًا مع دقة Type Hints التي تكتبها.

### تشغيل mypy على مشروع كامل

```bash
# فحص ملف واحد
uv run mypy example.py

# فحص مجلد كامل (كل ملفات .py بداخله)
uv run mypy src/

# فحص المشروع كله من المجلد الرئيسي
uv run mypy .
```

🧾 **للقراءة فقط** — أوامر `bash` تُنفَّذ من الطرفية، لا من الخلية.

### إعدادات mypy: ملف pyproject.toml

بدلًا من تمرير الخيارات عبر سطر الأوامر في كل مرة، يُفضَّل ضبط mypy مرة واحدة ضمن ملف `pyproject.toml` (نفس الملف الذي يدير مشروعك مع uv):

```toml
[tool.mypy]
python_version = "3.14"
disallow_untyped_defs = true      # يرفض الدوال بدون Type Hints إطلاقًا
disallow_incomplete_defs = true   # يرفض الدوال ذات Type Hints جزئية فقط
warn_return_any = true            # يُحذّر عند إرجاع Any من دالة ذات نوع إرجاع محدد
warn_unused_ignores = true        # يُحذّر عند استخدام # type: ignore بلا داعٍ
no_implicit_optional = true       # يمنع افتراض Optional تلقائيًا
strict_equality = true            # يتحقق من منطقية عمليات المقارنة ==
```

🧾 **للقراءة فقط** — محتوى ملف `toml`.

أو، بدلًا من ضبط كل خيار يدويًا، يمكنك تفعيل الوضع الصارم دفعة واحدة:

```toml
[tool.mypy]
python_version = "3.14"
strict = true
```

🧾 **للقراءة فقط** — محتوى ملف `toml`.

يُفعّل `strict = true` أكثر من عشرة خيارات تحقق صارمة دفعة واحدة. هذا هو الخيار الموصى به في المشاريع الجديدة التي تطمح لأعلى مستوى من الأمان.

> 💡 **نصيحة الخبير — التبني التدريجي:** إن كنت تعمل على مشروع كبير موجود مسبقًا بدون Type Hints، لا تُفعّل `strict = true` دفعة واحدة — فستُغرقك في مئات الأخطاء! ابدأ بخيارات بسيطة، وأضف الأنواع تدريجيًا ملفًا بملف، ثم فعّل الخيارات الصارمة تباعًا. صُممت Type Hints في Python أصلًا لتكون تدريجية (Gradual) لهذا السبب بالذات.

### تجاهل سطر معين عمدًا: `# type: ignore`

أحيانًا تعرف أن كودًا معينًا صحيح رغم اعتراض mypy عليه (مثلًا عند استخدام مكتبة خارجية بدون Type Hints جيدة). يمكنك تجاهل سطر محدد:

```python
import some_untyped_library  # type: ignore[import-untyped]

result = some_untyped_library.do_something()
```

🧾 **للقراءة فقط** — الوحدة `some_untyped_library` تُنشأ داخل مشروع محلي أو ليست حزمة عامة، فلا يمكن استيرادها هنا.

> ⚠️ **تحذير:** لا تستخدم `# type: ignore` كحل سريع لتجاوز خطأ لم تفهمه! اقرأ رسالة الخطأ أولًا وحاول فهمها وإصلاحها. استخدم `# type: ignore` فقط عندما تكون متأكدًا تمامًا من أن mypy مخطئ في هذه الحالة تحديدًا (وهذا نادر نسبيًا، ويحدث غالبًا بسبب نقص معلومات في مكتبة خارجية، لا بسبب خطأ منطقي حقيقي).

## 15.13 أفضل الممارسات العامة

**1. ابدأ من توقيعات الدوال، لا من كل متغير**

In [ ]:
# ✅ جيد: التركيز على توقيع الدالة (المعاملات والإرجاع)
def calculate_total(prices: list[float], tax_rate: float) -> float:
    subtotal = sum(prices)  # mypy يستنتج أن subtotal من نوع float
    return subtotal * (1 + tax_rate)

لا حاجة لكتابة `subtotal: float = sum(prices)` — الاستنتاج التلقائي (Type Inference) كافٍ هنا.

**2. تجنّب `Any` إلا للضرورة القصوى**

كل استخدام لـ `Any` نقطة عمياء في نظام الحماية لديك.

**3. استخدم `Optional`/`| None` بصراحة بدل الاعتماد على «التفاهم الضمني»**

```python
# ❌ غامض: هل يمكن أن يكون None؟ لا نعرف
def find(key: str) -> str:
    ...

# ✅ واضح: نعرف بالضبط أن النتيجة قد تكون None
def find(key: str) -> str | None:
    ...
```

⚠️ **للقراءة فقط** — خطأ مقصود للتوضيح، لذا لا يُنفَّذ.

**4. فعّل mypy منذ بداية المشروع (وليس في النهاية)**

كلما أضفت Type Hints منذ البداية، أصبحت العملية طبيعية وجزءًا من كتابة الكود، لا عبئًا إضافيًا لاحقًا.

**5. استخدم `reveal_type()` للتشخيص أثناء التطوير**

```python
x = [1, 2, 3]
reveal_type(x)  # أداة تشخيصية خاصة بـ mypy فقط
```

🧾 **للقراءة فقط** — مثال مقتطف يعتمد على `reveal_type` من مثال سابق.

```bash
uv run mypy example.py
```

🧾 **للقراءة فقط** — أوامر `bash` تُنفَّذ من الطرفية، لا من الخلية.

```
example.py:2: note: Revealed type is "builtins.list[builtins.int]"
```

هذه دالة وهمية خاصة بـ mypy فقط (لا تُستخدم فعليًا في الكود النهائي)، تُخبرك بالضبط عن النوع الذي استنتجته الأداة — مفيدة جدًا عند تصحيح مشاكل معقّدة.

### جدول مرجعي شامل: كل الأنواع التي تعلمناها

| الصيغة | المعنى | مثال |
|---|---|---|
| `int`, `float`, `str`, `bool` | أنواع أساسية | `age: int = 25` |
| `None` | لا شيء | `-> None` |
| `list[T]` | قائمة من نوع T | `list[int]` |
| `tuple[T1, T2]` | صندوق بأنواع محددة | `tuple[str, int]` |
| `tuple[T, ...]` | صندوق بعدد متغير من نفس النوع | `tuple[int, ...]` |
| `dict[K, V]` | قاموس | `dict[str, int]` |
| `set[T]` | مجموعة | `set[str]` |
| `T \| None` | اختياري (قد يكون None) | — |
| `T1 \| T2` | إمّا هذا أو ذاك | — |
| `Any` | أي نوع (بدون حماية) | `Any` |
| `Callable[[Args], Ret]` | دالة | `Callable[[int], str]` |
| `Literal[...]` | قيم محددة فقط | `Literal["a", "b"]` |
| `TypedDict` | قاموس بحقول معروفة | `class P(TypedDict): x: int` |
| `TypeVar` / `[T]` | نوع عام (Generic) | `def f[T](x: T) -> T` |
| `Protocol` | بروتوكول هيكلي | `class Drawable(Protocol): ...` |

## 15.14 مشروع تطبيقي: نظام إدارة مهام مُنمَّط بالكامل

لنطبّق كل ما تعلمناه في مشروع متكامل: نظام بسيط لإدارة المهام (Task Manager)، مع Type Hints كاملة ودقيقة، يجتاز فحص mypy في الوضع الصارم (`strict = true`) دون أي خطأ.

### المتطلبات

- كل مهمة لها: عنوان، أولوية (محدودة بقيم معينة فقط)، وحالة (معلّقة/قيد التنفيذ/منجزة).
- نظام يسمح بإضافة، البحث، تصفية، وتحديث حالة المهام.
- دعم Generic لأي نوع من «المستودعات» (Repositories) القابلة لإعادة الاستخدام.
- استخدام Protocol لتعريف «أي شيء قابل للعرض» (Displayable).

In [ ]:
"""
نظام إدارة المهام - مثال تطبيقي شامل على Type Hints
"""
from __future__ import annotations

from dataclasses import dataclass, field
from datetime import date
from typing import Literal, Protocol, Callable

# ---------------------------------------------------------
# 1. تعريف الأنواع الأساسية باستخدام Literal و Type Alias
# ---------------------------------------------------------
type Priority = Literal["منخفضة", "متوسطة", "عالية"]
type Status = Literal["معلّقة", "قيد التنفيذ", "منجزة"]

# ---------------------------------------------------------
# 2. Protocol لأي شيء "قابل للعرض"
# ---------------------------------------------------------
class Displayable(Protocol):
    def display(self) -> str:
        ...

# ---------------------------------------------------------
# 3. فئة المهمة (Task) باستخدام dataclass
# ---------------------------------------------------------
@dataclass
class Task:
    title: str
    priority: Priority = "متوسطة"
    status: Status = "معلّقة"
    due_date: date | None = None
    tags: list[str] = field(default_factory=list)

    def display(self) -> str:
        due_info = f" (موعد التسليم: {self.due_date})" if self.due_date else ""
        tags_info = f" [{', '.join(self.tags)}]" if self.tags else ""
        return f"📋 {self.title} - {self.priority} - {self.status}{due_info}{tags_info}"

    def mark_as(self, new_status: Status) -> None:
        self.status = new_status

# ---------------------------------------------------------
# 4. مستودع عام (Generic Repository) لإدارة أي نوع من العناصر
# ---------------------------------------------------------
class Repository[T]:
    """مستودع عام يدير مجموعة من العناصر من أي نوع."""

    def __init__(self) -> None:
        self._items: list[T] = []

    def add(self, item: T) -> None:
        self._items.append(item)

    def remove(self, item: T) -> None:
        self._items.remove(item)

    def all(self) -> list[T]:
        return self._items.copy()

    def filter(self, predicate: Callable[[T], bool]) -> list[T]:
        return [item for item in self._items if predicate(item)]

    def find_first(self, predicate: Callable[[T], bool]) -> T | None:
        for item in self._items:
            if predicate(item):
                return item
        return None

    def count(self) -> int:
        return len(self._items)

# ---------------------------------------------------------
# 5. مدير المهام (TaskManager) - يستخدم Repository
# ---------------------------------------------------------
class TaskManager:
    def __init__(self) -> None:
        self._repo: Repository[Task] = Repository()

    def add_task(
        self,
        title: str,
        priority: Priority = "متوسطة",
        due_date: date | None = None,
        tags: list[str] | None = None,
    ) -> Task:
        task = Task(
            title=title,
            priority=priority,
            due_date=due_date,
            tags=tags if tags is not None else [],
        )
        self._repo.add(task)
        return task

    def get_by_status(self, status: Status) -> list[Task]:
        return self._repo.filter(lambda t: t.status == status)

    def get_by_priority(self, priority: Priority) -> list[Task]:
        return self._repo.filter(lambda t: t.priority == priority)

    def find_task_by_title(self, title: str) -> Task | None:
        return self._repo.find_first(lambda t: t.title == title)

    def complete_task(self, title: str) -> bool:
        task = self.find_task_by_title(title)
        if task is None:
            return False
        task.mark_as("منجزة")
        return True

    def summary(self) -> dict[Status, int]:
        statuses: list[Status] = ["معلّقة", "قيد التنفيذ", "منجزة"]
        return {
            status: len(self.get_by_status(status))
            for status in statuses
        }

    def display_all(self) -> None:
        if self._repo.count() == 0:
            print("لا توجد مهام بعد.")
            return
        for task in self._repo.all():
            print(task.display())

# ---------------------------------------------------------
# 6. دالة عامة تعمل مع أي كائن Displayable (Protocol)
# ---------------------------------------------------------
def print_report(items: list[Displayable], title: str = "تقرير") -> None:
    print(f"=== {title} ===")
    for item in items:
        print(item.display())
    print(f"عدد العناصر: {len(items)}")

# ---------------------------------------------------------
# 7. برنامج تجريبي
# ---------------------------------------------------------
def main() -> None:
    manager = TaskManager()

    manager.add_task(
        "كتابة تقرير شهري",
        priority="عالية",
        due_date=date(2025, 6, 30),
        tags=["عمل", "عاجل"],
    )
    manager.add_task("قراءة كتاب Python", priority="منخفضة")
    manager.add_task(
        "مراجعة الكود مع الفريق",
        priority="متوسطة",
        tags=["فريق"],
    )

    print("كل المهام:")
    manager.display_all()
    print()

    manager.complete_task("قراءة كتاب Python")

    print("المهام العالية الأولوية:")
    for task in manager.get_by_priority("عالية"):
        print(task.display())
    print()

    print("ملخص الحالات:")
    for status, count in manager.summary().items():
        print(f"  {status}: {count}")
    print()

    high_priority_tasks: list[Displayable] = list(manager.get_by_priority("عالية"))
    print_report(high_priority_tasks, title="تقرير المهام العاجلة")

if __name__ == "__main__":
    main()

### فحص المشروع بـ mypy

```bash
uv run mypy task_manager.py --strict
```

🧾 **للقراءة فقط** — أوامر `bash` تُنفَّذ من الطرفية، لا من الخلية.

**الناتج المتوقع** (إن كُتب الكود بشكل صحيح تمامًا):

```
Success: no issues found in 1 source file
```

> 💡 **نصيحة الخبير:** لاحظ استخدام `from __future__ import annotations` في بداية الملف. هذا السطر (مفيد خصوصًا في إصدارات أقدم من Python) يجعل كل Type Hints تُعامَل كنصوص (Strings) أثناء التشغيل، فلا تُقيَّم فعليًا، مما يُسرّع بدء تشغيل البرنامج قليلًا ويحل مشاكل «التوقيعات الدائرية» (عندما تحتاج فئة أن تُشير لنفسها أو لفئة لاحقة في الكود). في Python 3.14 أصبح هذا السلوك قريبًا من الافتراضي في بعض الحالات، لكن إضافته صراحة لا تزال ممارسة جيدة ومتوافقة مع إصدارات أخرى.

## 15.15 تمارين

### 🟢 مستوى مبتدئ

**تمرين 1:** أضف Type Hints كاملة لهذه الدالة (المعاملات وقيمة الإرجاع):

In [ ]:
def calculate_area(width, height):
    return width * height

**تمرين 2:** اكتب دالة `get_grade(score: int) -> str` تُرجع تقدير الطالب («ممتاز»، «جيد»، «راسب») حسب الدرجة. استخدم `Literal` لنوع الإرجاع.

**تمرين 3:** ما الفرق بين `Optional[int]` و `int | None`؟ وأيهما يُفضَّل استخدامه في Python 3.14؟

<details>
<summary>✅ الحل النموذجي</summary>

In [ ]:
# تمرين 1
def calculate_area(width: float, height: float) -> float:
    return width * height

# تمرين 2
from typing import Literal

Grade = Literal["ممتاز", "جيد", "راسب"]

def get_grade(score: int) -> Grade:
    if score >= 90:
        return "ممتاز"
    elif score >= 60:
        return "جيد"
    else:
        return "راسب"

# تمرين 3
# لا فرق في المعنى: كلاهما يعني "قد يكون int أو None".
# Optional[int] هي الصيغة القديمة (تحتاج: from typing import Optional).
# int | None هي الصيغة الحديثة الموصى بها في Python 3.10+ (لا تحتاج استيراد).
# في كتابنا نستخدم دائمًا: int | None

</details>

### 🟡 مستوى متوسط

**تمرين 4:** صحّح الأخطاء التي سيكتشفها mypy في هذا الكود:

In [ ]:
# ⚠️ مثال يُظهر خطأً مقصودًا: نلتقطه هنا ليكتمل «تشغيل الكل»،
# وسترى نوع الخطأ ورسالته في المخرجات كما في الطرفية.
try:
    def find_product(product_id: int, catalog: dict[int, str]) -> str:
        return catalog.get(product_id)

    def get_discount(price: float, percent: int) -> float:
        return price - (price * percent)

    products = {1: "لابتوب", 2: "هاتف"}
    name = find_product(3, products)
    print(name.upper())
except (Exception, SystemExit) as _err:
    print(f"⚠️ {type(_err).__name__}: {_err}")

**تمرين 5:** اكتب فئة Generic باسم `Stack[T]` (مكدّس) تدعم:
- `push(item: T) -> None`
- `pop() -> T | None`
- `peek() -> T | None`
- `is_empty() -> bool`

ثم ثبّت mypy وتأكد أن الكود يجتاز الفحص بدون أخطاء.

**تمرين 6:** عرّف `Protocol` باسم `Comparable` يتطلب دالة `__lt__(self, other) -> bool`، ثم اكتب دالة `find_max[T: Comparable](items: list[T]) -> T` تُرجع أكبر عنصر في قائمة (دون استخدام `max()` الجاهزة).

<details>
<summary>✅ الحل النموذجي</summary>

In [ ]:
# ⚠️ مثال يُظهر خطأً مقصودًا: نلتقطه هنا ليكتمل «تشغيل الكل»،
# وسترى نوع الخطأ ورسالته في المخرجات كما في الطرفية.
try:
    # تمرين 4 - الكود المصحح
    def find_product(product_id: int, catalog: dict[int, str]) -> str | None:
        return catalog.get(product_id)  # قد تكون None فعلاً

    def get_discount(price: float, percent: int) -> float:
        return price - (price * (percent / 100))  # كانت تُطرح قيمة كبيرة جدًا خطأً منطقيًا

    products: dict[int, str] = {1: "لابتوب", 2: "هاتف"}
    name = find_product(3, products)

    if name is not None:
        print(name.upper())
    else:
        print("المنتج غير موجود")

    # تمرين 5
    class Stack[T]:
        def __init__(self) -> None:
            self._items: list[T] = []

        def push(self, item: T) -> None:
            self._items.append(item)

        def pop(self) -> T | None:
            if self.is_empty():
                return None
            return self._items.pop()

        def peek(self) -> T | None:
            if self.is_empty():
                return None
            return self._items[-1]

        def is_empty(self) -> bool:
            return len(self._items) == 0

    stack: Stack[int] = Stack()
    stack.push(1)
    stack.push(2)
    print(stack.pop())   # 2
    print(stack.peek())  # 1

    # تمرين 6
    from typing import Protocol

    class Comparable(Protocol):
        def __lt__(self, other: object) -> bool:
            ...

    def find_max[T: Comparable](items: list[T]) -> T:
        if not items:
            raise ValueError("القائمة فارغة")
        current_max = items[0]
        for item in items[1:]:
            if current_max < item:
                current_max = item
        return current_max

    print(find_max([3, 7, 2, 9, 4]))        # 9
    print(find_max(["بطة", "قطة", "أسد"]))  # قطة (حسب الترتيب الأبجدي)
except (Exception, SystemExit) as _err:
    print(f"⚠️ {type(_err).__name__}: {_err}")

**ملاحظة:** تعني `[T: Comparable]` أن «T أي نوع يُحقّق بروتوكول Comparable» — يُسمى هذا **Bounded TypeVar** (متغير نوع مقيّد).

</details>

### 🔴 تحدي

**تمرين 7:** صمّم نظام تخزين مؤقت (Cache) عام بالكامل:

1. فئة `Cache[K, V]` تخزّن أزواج مفتاح-قيمة من أي نوع.
2. دالة `get(key: K) -> V | None`.
3. دالة `set(key: K, value: V) -> None`.
4. دالة `get_or_compute(key: K, compute_fn: Callable[[], V]) -> V`: تُرجع القيمة إن وُجدت، وإلا تحسبها عبر `compute_fn`، تخزّنها، ثم تُرجعها.
5. استخدم `TypedDict` لتعريف إحصائيات الـ Cache (`hits: int`, `misses: int`) وأضف دالة `stats() -> CacheStats`.
6. شغّل mypy بوضع `--strict` وتأكد أن الكود خالٍ تمامًا من الأخطاء.

<details>
<summary>✅ الحل النموذجي</summary>

```python
from typing import TypedDict, Callable

class CacheStats(TypedDict):
    hits: int
    misses: int

class Cache[K, V]:
    def __init__(self) -> None:
        self._data: dict[K, V] = {}
        self._hits: int = 0
        self._misses: int = 0

    def get(self, key: K) -> V | None:
        if key in self._data:
            self._hits += 1
            return self._data[key]
        self._misses += 1
        return None

    def set(self, key: K, value: V) -> None:
        self._data[key] = value

    def get_or_compute(self, key: K, compute_fn: Callable[[], V]) -> V:
        existing = self.get(key)
        if existing is not None:
            return existing
        new_value = compute_fn()
        self.set(key, new_value)
        return new_value

    def stats(self) -> CacheStats:
        return {"hits": self._hits, "misses": self._misses}

def expensive_computation(n: int) -> int:
    print(f"🔄 جارِ الحساب الباهظ لـ {n} ...")
    return n ** 2

cache: Cache[int, int] = Cache()

result1 = cache.get_or_compute(5, lambda: expensive_computation(5))
print(result1)  # يطبع "جارِ الحساب..." ثم 25

result2 = cache.get_or_compute(5, lambda: expensive_computation(5))
print(result2)  # لا يطبع "جارِ الحساب" لأنها مخزّنة! يطبع 25 مباشرة

print(cache.stats())  # {'hits': 1, 'misses': 1}
```

🧾 **للقراءة فقط** — مثال مقتطف يعتمد على `K`، `V` من مثال سابق.

**تحقق بـ mypy:**

```bash
uv run mypy cache_example.py --strict
```

🧾 **للقراءة فقط** — أوامر `bash` تُنفَّذ من الطرفية، لا من الخلية.

```
Success: no issues found in 1 source file
```

</details>

## 15.16 اختبر فهمك (Quiz)

<details>
<summary>❓ السؤال 1: هل تمنع Python تشغيل الكود إن كانت الأنواع غير متطابقة مع Type Hints؟</summary>

لا. تتجاهل Python Type Hints تمامًا وقت التشغيل. هي «تلميحات» فقط موجَّهة للقارئ البشري ولأدوات خارجية مثل mypy والمحررات. يجب تشغيل أداة مثل mypy بشكل منفصل لاكتشاف هذه الأخطاء قبل التشغيل.

</details>

<details>
<summary>❓ السؤال 2: ما الفرق بين `list[int]` و `List[int]`؟</summary>

لا فرق في المعنى. `List[int]` هي الصيغة القديمة التي تتطلب `from typing import List`، بينما `list[int]` هي الصيغة الحديثة المدمجة في اللغة منذ Python 3.9، ولا تحتاج استيرادًا. نستخدم دائمًا `list[int]` في هذا الكتاب.

</details>

<details>
<summary>❓ السؤال 3: متى يجب استخدام `Any` بدل تحديد نوع دقيق؟</summary>

نادرًا جدًا، فقط عند التعامل مع بيانات غير معروفة الشكل فعليًا (كنتيجة لتحليل JSON خارجي)، أو أثناء ترحيل كود قديم تدريجيًا. يجب تجنّبها قدر الإمكان لأنها تُعطّل كل حماية نظام الأنواع على القيمة المعنية.

</details>

<details>
<summary>❓ السؤال 4: ما الفرق بين `Protocol` و `ABC` (Abstract Base Class)؟</summary>

تتطلب `ABC` وراثة صريحة (Nominal Typing) — يجب أن تُعلن الفئة أنها تَرث من الفئة المجردة. أما `Protocol` فلا يتطلب أي وراثة؛ يكفي أن تمتلك الفئة الدوال المطلوبة بنفس التوقيع (Structural Typing) — «إن كان شكلك مطابقًا، فأنت تحقق البروتوكول تلقائيًا»، بصرف النظر عن هرمية الوراثة.

</details>

<details>
<summary>❓ السؤال 5: ما وظيفة `TypeVar` أو الصيغة `[T]` في تعريف دالة؟</summary>

تسمح بكتابة دوال أو فئات عامة (Generic) تعمل مع أي نوع، مع الحفاظ على «الترابط» بين نوع المدخل ونوع المخرج. مثلًا، تضمن `def first[T](items: list[T]) -> T` أن ناتج الدالة هو نفس نوع عناصر القائمة المُمرَّرة، بصرف النظر عن النوع الفعلي.

</details>

<details>
<summary>❓ السؤال 6: لماذا نفضّل `str | None` على استخدام قيمة افتراضية وهمية مثل `""` للدلالة على «لا توجد قيمة»؟</summary>

لأن `""` نص صالح فعليًا وقد يكون قيمة حقيقية مقصودة (نص فارغ)، بينما يُعبّر `None` صراحة وبدون لبس عن «غياب القيمة تمامًا». استخدام `str | None` يجعل القصد واضحًا، ويجبرك (عبر mypy) على التحقق من `None` قبل استخدام القيمة، مما يمنع أخطاء شائعة.

</details>

## 15.17 ملخص الفصل (Cheat Sheet)

In [ ]:
# ===== الأنواع الأساسية =====
age: int = 25
price: float = 19.99
name: str = "سارة"
active: bool = True

# ===== الهياكل المركّبة =====
numbers: list[int] = [1, 2, 3]
point: tuple[float, float] = (1.0, 2.0)
scores: tuple[int, ...] = (1, 2, 3, 4)
ages: dict[str, int] = {"أحمد": 30}
tags: set[str] = {"a", "b"}

# ===== القيم الاختيارية والمركّبة =====
result: str | None = None          # اختياري (الصيغة الحديثة)
value: int | str = 5               # إمّا هذا أو ذاك

# ===== Any (استخدمها بحذر!) =====
from typing import Any
data: Any = {"whatever": "shape"}

# ===== Callable (الدوال كقيم) =====
from typing import Callable
operation: Callable[[int, int], int]

# ===== Literal (قيم محددة فقط) =====
from typing import Literal
Status = Literal["pending", "done"]

# ===== TypedDict (قاموس مُهيكَل) =====
from typing import TypedDict
class User(TypedDict):
    name: str
    age: int

# ===== Generics (البرمجة العامة) =====
def first[T](items: list[T]) -> T:
    return items[0]

class Box[T]:
    def __init__(self, content: T) -> None:
        self.content = content

# ===== Protocol (duck typing مُهيكَل) =====
from typing import Protocol
class Drawable(Protocol):
    def draw(self) -> str: ...

# ===== mypy =====
# التثبيت:
#   uv add --dev mypy
# التشغيل:
#   uv run mypy file.py
#   uv run mypy . --strict
# الإعداد في pyproject.toml:
#   [tool.mypy]
#   strict = true

| المفهوم | الاستخدام السريع |
|---|---|
| `T \| None` | قيمة قد تكون غائبة |
| `T1 \| T2` | أحد نوعين |
| `Any` | تعطيل التحقق (نادرًا) |
| `Callable[[Args], Ret]` | دالة كمعامل |
| `Literal[...]` | قيم ثابتة محددة |
| `TypedDict` | قاموس بحقول معروفة |
| `[T]` / `TypeVar` | نوع عام (Generic) |
| `Protocol` | duck typing منظم |
| `mypy` | أداة التحقق الساكن |

## 15.18 ماذا بعد؟

لقد أتقنت الآن أداة قوية جدًا تجعل كودك أكثر أمانًا ووضوحًا واحترافية: Type Hints المتقدمة وأداة mypy. من الآن فصاعدًا، سيستفيد كل كود تكتبه في بقية هذا الكتاب من هذا الفهم العميق — ستلاحظ توقيعات الدوال بعين جديدة، وستدرك لماذا اخترنا كتابتها بهذا الشكل بالذات منذ الفصل الثالث.

لكن حتى الآن، كانت كل برامجنا تعيش وتموت داخل الذاكرة فقط — بمجرد إغلاق البرنامج، تختفي كل البيانات. في الفصل القادم، سنتعلم كيف نحفظ البيانات بشكل دائم على القرص الصلب، وكيف نقرأها مرة أخرى: التعامل مع الملفات النصية، تنسيق JSON المنتشر في كل مكان (واجهات برمجية، إعدادات، تبادل بيانات)، ملفات CSV (الجداول)، ومكتبة `pathlib` الحديثة للتعامل الأنيق مع مسارات الملفات والمجلدات.

---

### الخطوة التالية

- [الفصل السادس عشر: الملفات وتنسيقات البيانات](https://colab.research.google.com/github/AhmedAlmaghz/Python-Book/blob/main/dist/notebooks/chapter-16.ipynb)
- [فهرس كل الدفاتر](https://colab.research.google.com/github/AhmedAlmaghz/Python-Book/blob/main/dist/notebooks/00-index.ipynb)

*الفصل الخامس عشر — من كتاب «بايثون 3.14: من الصفر إلى الاحتراف».*